## Jogos Olímpicos e Paralímpicos de Verão: Análise Comparativa e Socioeconômica do Desempenho Global

### Integrantes:
- Élia Dutra Sena
- Luis Fernando de Araujo Oliveira
- Maria Eloisa Silva de Sousa
- Maria Clara Caldas Fernandes

--- 

### Introdução:

Nosso objetivo com esse projeto é realizar uma análise comparativa e socioeconômica do desempenho das nações nas Olimpíadas e Paralimpíadas de Verão.  
De um modo geral, buscamos identificar países com eficiência socioeconômica no esporte, investigar o que pode levar um país a ter alguma vantagem e analisar sua evolução ao longo do tempo.

### Dados Usados:

## Base de Dados:
- **Dados esportivos:**
  
  Histórico de medalhas por ano: país e suas respectivas medalhas recebidas da olimpíada do ano.
  
  Histórico de medalhas geral: país e a quantidade total de medalhas adquiridas em todas as olimpíadas.
  
  Atletas (Kaggle): dados gerais (nome, gênero, idade, peso, altura, time, esporte, cidade, medalha) sobre todos os atletas que participaram das olimpíadas entre 1896-2016.

  
- **Dados socioeconômicos:**
  
  PIB mundial.
  
  IDH mundial.


  se usar os sem medlhas colocar

### Pré-processamento: 

In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler   
import scipy.stats as stats

##### Tratamento do CSV dos atletas:

Cria um dataframe que lê o arquivo com os dados dos atletas:

In [16]:
df = pd.read_csv('athlete_events.csv', encoding='ISO-8859-1')
display(df.head())

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


Organiza o arquivo tirando as linhas que tem atletas que competiram  em edições anteriores a de 1960 e nas olimpíadas de inverno e cria um novo CSV com os dados arrumados:

In [17]:
df_organizado = df[df['Year'] >= 1960].copy()
df_organizado = df_organizado[df_organizado["Season"] != "Winter"]

df_organizado.to_csv('athlete_events_limpo.csv', index=False, encoding='ISO-8859-1')
df_organizado.head()

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
31,12,Jyri Tapani Aalto,M,31.0,172.0,70.0,Finland,FIN,2000 Summer,2000,Summer,Sydney,Badminton,Badminton Men's Singles,NaN
32,13,Minna Maarit Aalto,F,30.0,159.0,55.5,Finland,FIN,1996 Summer,1996,Summer,Atlanta,Sailing,Sailing Women's Windsurfer,NaN
33,13,Minna Maarit Aalto,F,34.0,159.0,55.5,Finland,FIN,2000 Summer,2000,Summer,Sydney,Sailing,Sailing Women's Windsurfer,NaN


##### Tratamento dos arquivos de medalhas das Paralimpíadas e Olimpíadas :

**Olimpíadas:**

Trata o nome dos países, pois estava diferente do que tem no PIB e dava erro ao tentar juntar os datasets.

    Como era: Brasil (BRA)
    Após o tratamento: fica uma coluna para o país que tem apenas o nome: Brasil e cria uma nova coluna que recebe o NOC: BRA.

Organizamos a posição dos países para que os que mais têm medalhas fiquem no topo.

Após tratar, cria um novo CSV.

In [28]:
df_olimpiadas = pd.read_csv('medalhas_olimpiadas_wikipedia.csv', encoding='utf-8')
display(df_olimpiadas.head())

,No.,País,Jogos,Ouro,Prata,Bronze,Total
0,1,USA Estados Unidos,29,1101,880,781,2762
1,2,URS União Soviética,9,395,319,296,1010
2,3,CHN China,12,302,227,197,726
3,4,GBR Grã-Bretanha,30,300,338,344,982
4,5,FRA França,30,240,280,298,818


In [ ]:
df_olimpiadas = df_olimpiadas.sort_values(by='Total', ascending=False)
display(df_olimpiadas.head())


,No.,País,Jogos,Ouro,Prata,Bronze,Total
0,1,USA Estados Unidos,29,1101,880,781,2762
1,2,URS União Soviética,9,395,319,296,1010
3,4,GBR Grã-Bretanha,30,300,338,344,982
4,5,FRA França,30,240,280,298,818
2,3,CHN China,12,302,227,197,726
...,...,...,...,...,...,...,...
150,147,ERI Eritreia,6,0,0,1,1
151,147,GUY Guiana,18,0,0,1,1
152,147,IRQ Iraque,15,0,0,1,1
153,147,MRI Maurício,10,0,0,1,1


In [ ]:
df_olimpiadas_limpo = df_olimpiadas['País'].str.extract(r'^([A-Z]{3})\s+(.*)')
df_olimpiadas_limpo.columns = ['NOC', 'País_Limpo']

df_olimpiadas['NOC'] = df_olimpiadas_limpo['NOC']
df_olimpiadas['País'] = df_olimpiadas_limpo['País_Limpo']

df_olimpiadas['País'] = df_olimpiadas['País'].str.replace(r'\[.*\]|\*', '', regex=True).str.strip()

display(df_olimpiadas.head())

,No.,País,Jogos,Ouro,Prata,Bronze,Total,NOC
0,1,Estados Unidos,29,1101,880,781,2762,USA
1,2,União Soviética,9,395,319,296,1010,URS
3,4,Grã-Bretanha,30,300,338,344,982,GBR
4,5,França,30,240,280,298,818,FRA
2,3,China,12,302,227,197,726,CHN


In [31]:
df_olimpiadas.to_csv('olimpiadas_medalhas_arrumado.csv', index=False, encoding='utf-8')

**Paralimpíadas:**

Trata o nome dos países, pois estava diferente do que tem no PIB e dava erro ao tentar juntar os datasets.

    Como era: Brasil (BRA)
    Após o tratamento: fica uma coluna para o país que tem apenas o nome: Brasil e cria uma nova coluna que recebe o NOC: BRA.

Além disso, Unifica as Alemanhas que tinham 3 versões diferentes, Alemanha (GER) [a], Alemanha Ocidental (FRG) [b], Alemanha Oriental (GDR) [c], somando suas medalhas.

Ordenamos o dataframe para que os maiores ganhadores de medalhas fiquem no topo.

Ao analisar dataframe nessa nova ordem percebemos que tinha uma linha que somava todas as outras medalhas juntas chamada "Totais", a tiramos para manter apenas países.

Após tratar, cria um novo CSV.

In [32]:
df_paraolimpiadas = pd.read_csv('medalhas_paralimpiadas_wikipedia.csv', encoding='utf-8')
display(df_paraolimpiadas.head())

,País,№,Ouro,Prata,Bronze,Total
0,África do Sul (RSA),11,121,95,88,304
1,Alemanha (GER) [a],8,199,266,253,718
2,Alemanha Ocidental (FRG) [b],8,322,260,246,828
3,Alemanha Oriental (GDR) [c],1,0,3,1,4
4,Angola (ANG),6,4,3,1,8


In [33]:
df_paraolimpiadas['NOC'] = df_paraolimpiadas['País'].str.extract(r'\((.*?)\)')

df_paraolimpiadas['País'] = df_paraolimpiadas['País'].str.replace(r'\s*\(.*', '', regex=True)
df_paraolimpiadas['País'] = df_paraolimpiadas['País'].str.replace(r'\[.*\]', '', regex=True).str.strip()
display(df_paraolimpiadas.head())

,País,№,Ouro,Prata,Bronze,Total,NOC
0,África do Sul,11,121,95,88,304,RSA
1,Alemanha,8,199,266,253,718,GER
2,Alemanha Ocidental,8,322,260,246,828,FRG
3,Alemanha Oriental,1,0,3,1,4,GDR
4,Angola,6,4,3,1,8,ANG


In [34]:
alemanhas = ['Alemanha Ocidental', 'Alemanha Oriental']
df_paraolimpiadas['País'] = df_paraolimpiadas['País'].replace(alemanhas, 'Alemanha')

df_paraolimpiadas = df_paraolimpiadas.groupby('País').agg({
    'Ouro': 'sum',
    'Prata': 'sum',
    'Bronze': 'sum',
    'Total': 'sum',
    'NOC': 'first' 
}).reset_index()

df_paraolimpiadas = df_paraolimpiadas.sort_values(by='Total', ascending=False)

display(df_paraolimpiadas.head())

,País,Ouro,Prata,Bronze,Total,NOC
115,Totais,7513,7298,7336,22147,NaN
40,Estados Unidos,808,736,739,2283,USA
48,Grã-Bretanha,667,621,626,1914,GBR
0,Alemanha,521,529,500,1550,GER
22,China,535,400,302,1237,CHN


In [35]:
df_paraolimpiadas = df_paraolimpiadas[df_paraolimpiadas['País'] != 'Totais']

display(df_paraolimpiadas.head())

,País,Ouro,Prata,Bronze,Total,NOC
40,Estados Unidos,808,736,739,2283,USA
48,Grã-Bretanha,667,621,626,1914,GBR
0,Alemanha,521,529,500,1550,GER
22,China,535,400,302,1237,CHN
6,Austrália,389,422,394,1205,AUS


In [36]:
df_paraolimpiadas.to_csv('paralimpiada_medalhas_arrumado.csv', index=False, encoding='utf-8')

### Perguntas:

#### 1 - O "Efeito Casa" 

​**A Pergunta:** O país sede realmente ganha mais medalhas quando sedia o evento?  

​Comparação: Analisar o desempenho dos países sedes na olimpíada anterior, na que sediaram e na posterior.  


#### 2 - Representatividade Feminina  
**A ​Pergunta:** Como a participação das mulheres evoluiu ao longo das décadas? Existem esportes que ainda são muito desiguais?  

​Comparação: Duas linhas do tempo (1960–2024). Uma mostrando a % de mulheres nas Olimpíadas e outra nas Paralimpíadas.  
​Análise: As Paralimpíadas começaram depois, mas será que evoluíram mais rápido na inclusão feminina?



#### 3 - Monopólios Esportivos
**​A Pergunta:** Quais esportes são dominados por um único país?  

​Comparação: Analisar a porcentagem de medalhas de ouro ganhas por um único país em modalidades específicas.  
​Ex: Tênis de Mesa (China), Basquete Masculino (EUA), Corridas de Longa Distância (Quênia/Etiópia), Tiro com Arco (Coreia do Sul).


#### 4 - Eficiência por PIB
​**A Pergunta:** O dinheiro "compra" mais medalhas em qual dos dois eventos?  

​Comparação: Cruzar o PIB dos países com o quadro de medalhas de ambos os eventos.  
​Hipótese: Equipamentos paralímpicos (cadeiras de rodas de titânio, próteses de fibra de carbono) são caríssimos. A correlação entre PIB e vitória é mais forte nas Paralimpíadas do que nas Olimpíadas


#### 5 - A "Correlação de Potência" (O Espelho Distorcido)  

**​A Pergunta:** Os países que dominam as Olimpíadas são os mesmos que dominam as Paralimpíadas?  

Comparação: O quanto é possível correlacionar o desempenho nas Olimpíadas com o desempenho nas Paralimpíadas? O número de medalhas segue numa mesma proporção?  
Análise: Gráficos de barra comparativos e um gráfico de dispersão (Scatter Plot) onde o eixo X são medalhas Olímpicas e o eixo Y são medalhas Paralímpicas.


### Conclusão:

// Resumir os resultados encontrados e explicar por que são importantes. Apontar limitações, trabalhos futuros e melhorias que podem ser feitas.

OBS: Pelo menos três tipos diferentes de gráficos devem ser usados no relatório.